# QLoRA Fine-Tuning FLAN-T5-base for Table-to-Text Summarization (ToTTo)

**Goal:** fine-tune `google/flan-t5-base` with quantized LoRA to generate a single factual sentence
summarizing the highlighted cells of a Wikipedia table (ToTTo dataset).

**Pipeline:** raw ToTTo JSON -> prompt/target pairs -> tokenization ->Quantization -> LoRA fine-tuning -> save adapter -> inference & evaluation.

**Hardware:** local RTX 3060 (12GB VRAM), Ryzen 7 5600X, 32GB RAM - all training done on-device, no cloud.


## 1. Environment Setup

Confirm CUDA / GPU availability before doing anything expensive.

In [1]:
# GPU setup
import torch

# Check CUDA availability and GPU properties
print(f"torch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch version: 2.11.0+cu128
CUDA version: 12.8
CUDA available: True
GPU count: 1
GPU name: NVIDIA GeForce RTX 3060
VRAM: 12.9 GB


## 2. Base Model & Tokenizer Loading

In [ ]:
# Load the model and tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import torch
save_directory = "./models/flan-t5-base-local"

# Define the quantization configuration for 4-bit quantization
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,                     # Enable 4-bit quantization
        bnb_4bit_quant_type="nf4",             # Use NF4 quantization type
        bnb_4bit_compute_dtype=torch.bfloat16, # Compute in bfloat16 for stability
        bnb_4bit_use_double_quant=True,        # Nested quantization for extra savings
    )
# Load from local directory
tokenizer = AutoTokenizer.from_pretrained(save_directory, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    save_directory,
    quantization_config=bnb_config,   # 4 bit quantization will be used in qlora
    device_map="auto",           # Automatically place model layers
    trust_remote_code=True,      # Required for some models
    dtype=torch.bfloat16,        # Use bfloat16 for non-quantized parts
)


c:\Users\Youssef\Desktop\slm_fine_tune\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0714 22:36:46.857000 2976 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]c:\Users\Youssef\Desktop\slm_fine_tune\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 282/282 [00:01<00:00, 269.10it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will N

## 3. Dataset Loading & Tokenization

In [4]:
# Load the prompt datasets
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "totto_data/train.jsonl",
        "validation": "totto_data/validation.jsonl"
    }
)

print(dataset["train"][0])
print(dataset["validation"][0])

{'id': 1762238357686640028, 'prompt': 'Task:\nGenerate a single factual sentence describing the information contained in the highlighted cells.\nUse only information provided below.\nDo not invent facts.\nPage Title:\nList of 8/9 PM telenovelas of Rede Globo\nSection Title:\n2000s\nHighlighted Cells:\nRow: 13, Column: 2\nTable:\n#\tRun\tTitle\tChapters\tAuthor\tDirector\tIbope Rating\n59\tJune 5, 2000— February 2, 2001\tLaços de Família\t209\tManoel Carlos\tRicardo Waddington\t44.9\n60\tFebruary 5, 2001— September 28, 2001\tPorto dos Milagres\t203\tAguinaldo Silva Ricardo Linhares\tMarcos Paulo Simões\t44.6\n61\tOctober 1, 2001— June 14, 2002\tO Clone\t221\tGlória Perez\tJayme Monjardim\t47.0\n62\tJune 17, 2002— February 14, 2003\tEsperança\t209\tBenedito Ruy Barbosa\tLuiz Fernando\t37.7\n63\tFebruary 17, 2003— October 10, 2003\tMulheres Apaixonadas\t203\tManoel Carlos\tRicardo Waddington\t46.6\n64\tOctober 13, 2003— June 25, 2004\tCelebridade\t221\tGilberto Braga\tDennis Carvalho\t46.

In [12]:
def preprocess(examples):
    model_inputs = tokenizer(
        examples["prompt"],
        max_length=384,
        truncation=True,
    )

    labels = tokenizer(
        text_target=examples["target"],
        max_length=64,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [13]:
tokenized_dataset = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names
)

In [6]:
# Sanity check: print the first tokenized example
print(tokenized_dataset["train"][0])
print(tokenized_dataset["validation"][0])
# print stats 
print(f"Number of training samples: {len(tokenized_dataset['train'])}")
print(f"Number of validation samples: {len(tokenized_dataset['validation'])}")

{'input_ids': [16107, 10, 6939, 2206, 3, 9, 712, 685, 3471, 7142, 3, 16012, 8, 251, 6966, 16, 8, 12566, 2640, 5, 2048, 163, 251, 937, 666, 5, 531, 59, 16, 2169, 6688, 5, 5545, 11029, 10, 6792, 13, 505, 87, 1298, 3246, 3, 1931, 5326, 15, 521, 7, 13, 1624, 15, 9840, 115, 32, 5568, 11029, 10, 2766, 7, 16388, 15, 26, 7845, 7, 10, 11768, 10, 10670, 29926, 10, 204, 4398, 10, 1713, 7113, 11029, 8647, 7, 10236, 2578, 27, 115, 32, 855, 21662, 3, 3390, 1515, 7836, 2766, 318, 2083, 3547, 4402, 325, 24065, 7, 20, 1699, 51, 2, 40, 23, 9, 460, 1298, 1140, 32, 15, 40, 19783, 2403, 6043, 32, 3129, 30557, 314, 27336, 1640, 2083, 7836, 4402, 318, 1600, 13719, 4402, 3625, 32, 103, 7, 8573, 9, 11176, 3, 23330, 71, 17996, 138, 26, 32, 26551, 2403, 6043, 32, 6741, 3272, 15, 7, 16902, 7, 1838, 32, 6619, 2, 15, 7, 314, 25652, 3, 4241, 1797, 1914, 4402, 318, 1515, 11363, 4407, 411, 4779, 782, 204, 2658, 350, 40, 4922, 52, 23, 9, 1915, 457, 9373, 526, 2963, 5670, 26, 603, 314, 26346, 3, 4056, 1515, 12864, 4407,

## 4. QLoRA Configuration

In [7]:
# LoRA configuration
from peft import LoraConfig, TaskType, get_peft_model
peft_config = LoraConfig(
    r=8,                                # Rank of the low-rank matrices
    lora_alpha=16,                      # Scaling factor for the low-rank matrices
    target_modules=["q", "v"],          # Target modules for LoRA
    lora_dropout=0.1,                   # Dropout rate for LoRA layers
    bias="none",                        # No bias in LoRA layers
    task_type=TaskType.SEQ_2_SEQ_LM,    # Task type for encoder-decoder models
    )

model = get_peft_model(model, peft_config)

In [8]:
model.print_trainable_parameters()

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


## 5. Training Configuration & Sanity Checks
### 5.1 Trainer Setup

In [23]:
import os
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback, DataCollatorForSeq2Seq
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    gradient_checkpointing=False,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=500,
    lr_scheduler_type="cosine",

    # Logging
    logging_steps=1,
    save_steps=1000,
    eval_steps=500,
    eval_strategy="epoch",
    save_strategy="epoch",
    train_sampling_strategy="group_by_length",
    logging_first_step=True,   
    logging_nan_inf_filter=False,

    # Generation
    predict_with_generate=False,
    generation_max_length=32,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Mixed precision
    fp16=False,
    bf16=True, 

    # Optimizer
    optim="adamw_torch", 

    seed=67,        # Set a random seed for reproducibility

    report_to="tensorboard",
)


data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
    

### 5.2 Pre-Training Sanity Check

Before committing to a full training run, confirm the pipeline produces a finite, reasonable loss on an untouched batch.

In [10]:
model.eval()
batch = next(iter(trainer.get_train_dataloader()))
batch = {k: v.to(model.device) for k, v in batch.items()}
with torch.no_grad():
    out = model(**batch)
print("loss on a fresh, untrained batch:", out.loss)

loss on a fresh, untrained batch: tensor(2.4219, device='cuda:0', dtype=torch.bfloat16)


## 6. Fine-Tuning

In [24]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.135812,1.613521
2,1.028346,1.575107
3,1.122873,1.568719


TrainOutput(global_step=22644, training_loss=1.8212749656514216, metrics={'train_runtime': 10294.7161, 'train_samples_per_second': 35.191, 'train_steps_per_second': 2.2, 'total_flos': 1.4621934698984448e+17, 'train_loss': 1.8212749656514216, 'epoch': 3.0})

In [25]:
# Save the fine-tuned model
model.save_pretrained("./models/flan-t5-base-totto-qlora-finetuned")

## 7. Evaluation & Inference
### 7.1 Reload Base Model + QLoRA Adapter


In [1]:
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

base_model = AutoModelForSeq2SeqLM.from_pretrained("./models/flan-t5-base-local", device_map="auto", trust_remote_code=True, dtype=torch.bfloat16)
fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    "./models/flan-t5-base-totto-qlora-finetuned"
)

tokenizer = AutoTokenizer.from_pretrained("./models/flan-t5-base-local")

fine_tuned_model.eval()
fine_tuned_model.cuda()

c:\Users\Youssef\Desktop\slm_fine_tune\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2150.73it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
W0715 02:08:06.779000 3888 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32128, 768)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32128, 768)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=8, out_features=768, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
               

### 7.2 Generate on Validation Set

In [2]:
from torch.utils.data import DataLoader

def generate_predictions(model, dataset, tokenizer, batch_size=96, max_new_tokens=64, num_beams=4):
    """Runs beam-search generation over `dataset` and returns (predictions, references)."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    predictions, references = [], []
    model.eval()

    for batch in loader:
        inputs = tokenizer(
            batch["prompt"],
            padding=True,
            truncation=True,
            max_length=384,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                do_sample=False,
            )

        preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(preds)
        references.extend(batch["target"])

    return predictions, references


In [5]:
ft_predictions, ft_references = generate_predictions(
    fine_tuned_model, dataset["validation"], tokenizer
)
print(f"Generated {len(ft_predictions)} predictions with the fine-tuned model.")


Generated 7700 predictions with the fine-tuned model.


In [6]:
import gc

# Free VRAM held by the fine-tuned model before loading a second full model
fine_tuned_model.to("cpu")
gc.collect()
torch.cuda.empty_cache()

base_model_for_eval = AutoModelForSeq2SeqLM.from_pretrained(
    "./models/flan-t5-base-local",
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
base_model_for_eval.eval()

base_predictions, base_references = generate_predictions(
    base_model_for_eval, dataset["validation"], tokenizer
)
print(f"Generated {len(base_predictions)} predictions with the base (non-fine-tuned) model.")

# References are identical for both models (same dataset, same order) 
assert base_references == ft_references


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1408.72it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generated 7700 predictions with the base (non-fine-tuned) model.


In [ ]:
import evaluate
import nltk

# METEOR needs these NLTK resources the first time it runs
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
meteor_metric = evaluate.load("meteor")
bertscore_metric = evaluate.load("bertscore")



[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Youssef\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Youssef\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Youssef\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [8]:
def compute_all_metrics(predictions, references, label="model"):
    """Computes ROUGE-L, BLEU, METEOR, BERTScore-F1, and (optionally) BLEURT for one set
    of generated predictions against their references."""

    def sanitize(texts, placeholder="[empty]"):
        # bert_score (and some metric libs) crash or misbehave on empty/whitespace-only
        # strings depending on the installed transformers version. Empty generations are
        # also a real signal (esp. from a zero-shot base model giving up early) worth
        # counting rather than silently dropping.
        cleaned, n_empty = [], 0
        for t in texts:
            if t is None or t.strip() == "":
                cleaned.append(placeholder)
                n_empty += 1
            else:
                cleaned.append(t)
        return cleaned, n_empty

    predictions, n_empty_preds = sanitize(predictions)
    references, n_empty_refs = sanitize(references)
    if n_empty_preds or n_empty_refs:
        print(
            f"[{label}] sanitized {n_empty_preds} empty prediction(s) and "
            f"{n_empty_refs} empty reference(s) before scoring."
        )

    refs_list = [[r] for r in references]  # bleu/meteor expect a list of references per example

    rouge_res = rouge_metric.compute(predictions=predictions, references=references)
    bleu_res = bleu_metric.compute(predictions=predictions, references=refs_list)
    meteor_res = meteor_metric.compute(predictions=predictions, references=references)
    bertscore_res = bertscore_metric.compute(
        predictions=predictions, references=references, lang="en"
    )

    results = {
        "model": label,
        "ROUGE-L": rouge_res["rougeL"],
        "BLEU": bleu_res["bleu"],
        "METEOR": meteor_res["meteor"],
        "BERTScore-F1": sum(bertscore_res["f1"]) / len(bertscore_res["f1"]),
    }

    return results


In [10]:
import pandas as pd

base_results = compute_all_metrics(base_predictions, base_references, label="Base (no fine-tuning)")
ft_results = compute_all_metrics(ft_predictions, ft_references, label="Fine-tuned (QLoRA)")

results_df = pd.DataFrame([base_results, ft_results]).set_index("model")
results_df


[Base (no fine-tuning)] sanitized 4 empty prediction(s) and 0 empty reference(s) before scoring.
[Fine-tuned (QLoRA)] sanitized 1 empty prediction(s) and 0 empty reference(s) before scoring.


,ROUGE-L,BLEU,METEOR,BERTScore-F1
model,,,,
Base (no fine-tuning),0.191407,0.046260,0.210843,0.855862
Fine-tuned (QLoRA),0.381938,0.171166,0.410853,0.903964
